# ДЗ 6.1. Перенос импульса по кольцу

Рассматривается перенос примеси с постоянной скоростью по периодической области:
$$
u_t+c u_x=0,\qquad x\in[0,1),\qquad c=1.
$$
Точное решение — сдвиг начального профиля без изменения формы:
$$
u(x,t)=u_0((x-ct)\bmod 1).
$$
Период обращения равен $T=1$. Сначала сравним схемы upwind и Лакса–Вендроффа
на гладком профиле $u_0(x)=1+\tfrac12\sin(2\pi x)$, затем — на прямоугольном
импульсе $u_0(x)=1$ при $0.2\le x<0.4$ и $0$ вне этого интервала.

Для гладкого профиля используем $N=50,100,200$ и $C=0.8$; для прямоугольного —
те же сетки и $C=0.5,0.8,1$. Неустойчивый случай $C=1.1$ рассмотрим отдельно.
Результаты сравниваются в одинаковые моменты $t=1$ и $t=5$.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, Markdown, display

plt.rcParams.update({
    "figure.dpi": 110,
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.22,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "animation.embed_limit": 40,
})

BLUE, ORANGE, GREEN = "#0072B2", "#D55E00", "#009E73"
METHODS = ("upwind", "LW")
LABELS = {"upwind": "Upwind", "LW": "Лакс–Вендрофф"}
COLORS = {"upwind": BLUE, "LW": ORANGE}
NS = (50, 100, 200)
CS = (0.5, 0.8, 1.0)
TIMES = (1.0, 5.0)


def table(headers, rows):
    clean = lambda value: str(value).replace("|", "&#124;")
    lines = ["| " + " | ".join(map(clean, headers)) + " |",
             "| " + " | ".join(["---"] * len(headers)) + " |"]
    lines += ["| " + " | ".join(map(clean, row)) + " |" for row in rows]
    display(Markdown("\n".join(lines)))

## 1. Сетка и численные схемы

На кольце берём узлы $x_j=j\Delta x$, $j=0,\ldots,N-1$, $\Delta x=1/N$.
Точка $x=1$ не включается. Индексы соседей понимаются по модулю $N$.
Обозначим $\nu_n=c\Delta t_n/\Delta x$ — число Куранта на текущем шаге.

При $c>0$ схема **upwind** имеет вид
$$
u_j^{n+1}=u_j^n-\nu_n(u_j^n-u_{j-1}^n).
$$
При $0\le\nu_n\le1$ это выпуклая комбинация двух значений: новые минимумы
и максимумы не возникают. Разрывы при этом размываются. На гладком решении
схема имеет первый порядок точности при фиксированном числе Куранта.

Схема **Лакса–Вендроффа**:
$$
u_j^{n+1}=u_j^n-\frac{\nu_n}{2}(u_{j+1}^n-u_{j-1}^n)
+\frac{\nu_n^2}{2}(u_{j+1}^n-2u_j^n+u_{j-1}^n).
$$
На гладком решении она имеет второй порядок. Обе схемы устойчивы при
$|\nu_n|\le1$, однако схема Лакса–Вендроффа не сохраняет монотонность.

В каждом опыте основной шаг равен $\Delta t=C\Delta x/c$. Последний шаг
до $t=1$ и до $t=5$ при необходимости сокращается; основное $C$ при
сгущении сетки сохраняется. Во всех схемах новое состояние вычисляется
целиком из предыдущего слоя.

Для проверки используем
$$
E_1(t)=\Delta x\sum_j|u_j(t)-u(x_j,t)|,\qquad
M(t)=\Delta x\sum_j u_j(t).
$$
Это узловые значения и узловая ошибка, а не средние по ячейкам.
Суммы периодических разностей равны нулю, поэтому обе схемы сохраняют
$M$ в точной арифметике. Здесь $M(0)=1$ для синусоиды и $M(0)=0.2$
для импульса на всех выбранных сетках.

In [ ]:
def initial_profile(x, profile):
    if profile == "smooth":
        return 1.0 + 0.5 * np.sin(2.0 * np.pi * x)
    if profile == "pulse":
        # Устраняем только округление координат на концах полуинтервала.
        x = np.asarray(x).copy()
        for edge in (0.2, 0.4):
            x = np.where(np.abs(x - edge) < 1e-12, edge, x)
        return ((x >= 0.2) & (x < 0.4)).astype(float)
    raise ValueError("Неизвестный профиль")


def exact_solution(x, t, profile):
    return initial_profile(np.mod(x - t, 1.0), profile)


def step(u, nu, method):
    left = np.roll(u, 1)
    if method == "upwind":
        return u - nu * (u - left)
    if method == "LW":
        right = np.roll(u, -1)
        return u - 0.5 * nu * (right - left) + 0.5 * nu**2 * (right - 2*u + left)
    raise ValueError("Неизвестная схема")


def time_grid(N, C, targets):
    """Основной шаг C/N; точное попадание в каждый заданный момент."""
    dt = C / N
    parts = [np.array([0.0])]
    start = 0.0
    for stop in targets:
        count = int(np.floor((stop - start) / dt + 1e-12))
        inner = start + dt * np.arange(1, count + 1)
        inner = inner[inner < stop - 1e-12]
        parts.extend([inner, np.array([float(stop)])])
        start = stop
    return np.concatenate(parts)


def simulate(N, C, profile, targets=TIMES):
    x = np.arange(N) / N
    t = time_grid(N, C, targets)
    result = {}
    for method in METHODS:
        states = np.empty((len(t), N))
        states[0] = initial_profile(x, profile)
        for k, dt in enumerate(np.diff(t)):
            states[k + 1] = step(states[k], N * dt, method)
        exact = exact_solution(x[None, :], t[:, None], profile)
        result[method] = {
            "x": x, "t": t, "u": states,
            "E1": np.mean(np.abs(states - exact), axis=1),
            "M": np.mean(states, axis=1),
            "min": np.min(states, axis=1),
            "max": np.max(states, axis=1),
        }
    return result


def at_time(run, t):
    k = np.flatnonzero(np.isclose(run["t"], t, rtol=0, atol=1e-12))
    if len(k) != 1:
        raise ValueError("Заданный момент отсутствует или повторяется")
    return k[0]

## 2. Гладкий профиль

После целого числа оборотов точное решение совпадает с начальным профилем.
На графиках ниже чёрный пунктир показывает точное решение, синий цвет —
upwind, оранжевый — схему Лакса–Вендроффа. Масштабы всех панелей одинаковы.

In [ ]:
smooth = {}
for N in NS:
    pair = simulate(N, 0.8, "smooth")
    for method in METHODS:
        smooth[method, N] = pair[method]

fig, axes = plt.subplots(2, 3, figsize=(13, 6.6), sharex=True, sharey=True,
                         layout="constrained")
xf = np.linspace(0, 1, 1001, endpoint=False)
for row, target in enumerate(TIMES):
    for col, N in enumerate(NS):
        ax = axes[row, col]
        ax.plot(xf, exact_solution(xf, target, "smooth"), "k--", lw=1.5,
                label="Точное решение")
        for method in METHODS:
            run = smooth[method, N]
            k = at_time(run, target)
            ax.plot(run["x"], run["u"][k], color=COLORS[method], label=LABELS[method])
        ax.set(title=f"N = {N}, t = {target:g}", xlim=(0, 1), ylim=(0.4, 1.6))
        if row == 1:
            ax.set_xlabel("x")
        if col == 0:
            ax.set_ylabel("u")
axes[0, 0].legend(fontsize=8)
fig.suptitle("Гладкий профиль, C = 0.8")
plt.show()

### Сходимость и сохранение количества примеси

Оценим порядок по двум соседним сеткам:
$$
p=\log_2\frac{E_1(N)}{E_1(2N)}.
$$
В таблице $p$ относится к переходу от предыдущей, более грубой сетки к текущей.
На логарифмических графиках пунктирные линии показывают зависимости
$E_1\sim\Delta x$ и $E_1\sim\Delta x^2$, привязанные к ошибке при $N=200$.

In [ ]:
rows = []
for target in TIMES:
    for method in METHODS:
        previous = None
        for N in NS:
            run = smooth[method, N]
            k = at_time(run, target)
            error = run["E1"][k]
            order = "—" if previous is None else f"{np.log2(previous / error):.3f}"
            drift = np.max(np.abs(run["M"] - run["M"][0]))
            rows.append([f"{target:g}", LABELS[method], N, f"{1/N:.3f}",
                         f"{error:.6e}", order, f"{drift:.2e}"])
            previous = error
table(["t", "Схема", "N", "Δx", "E₁", "p", "max |M − M₀|, 0 ≤ t ≤ 5"], rows)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4), layout="constrained")
dx = 1 / np.array(NS, dtype=float)
for ax, target in zip(axes, TIMES):
    for method, power in zip(METHODS, (1, 2)):
        errors = np.array([smooth[method, N]["E1"][at_time(smooth[method, N], target)]
                           for N in NS])
        ax.loglog(dx, errors, "o-", color=COLORS[method], label=LABELS[method])
        ax.loglog(dx, errors[-1] * (dx / dx[-1])**power, ":",
                  color=COLORS[method], label=f"наклон {power}")
    ax.set(xlabel="Δx", ylabel="E₁", title=f"t = {target:g}")
    ax.legend(fontsize=9)
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(11.5, 6), sharex=True, layout="constrained")
for col, method in enumerate(METHODS):
    for N in NS:
        run = smooth[method, N]
        axes[0, col].plot(run["t"], run["M"], label=f"N = {N}")
        axes[1, col].plot(run["t"], run["M"] - run["M"][0], label=f"N = {N}")
    axes[0, col].axhline(1, color="black", ls="--", lw=1, label="M₀ = 1")
    axes[0, col].set(title=LABELS[method], ylabel="M", ylim=(0.999, 1.001))
    axes[0, col].legend(fontsize=8, ncol=2)
    axes[1, col].set(xlabel="t", ylabel="M − M₀")
    axes[1, col].ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
fig.suptitle("Количество примеси и отклонение от начального значения, C = 0.8")
plt.show()

**Означает ли сохранение количества примеси правильность её распределения?**

Нет. Сумма значений может оставаться прежней, хотя сам профиль меняется.
У upwind амплитуда синусоиды уменьшается, особенно за пять оборотов на грубой
сетке. Схема Лакса–Вендроффа заметно лучше сохраняет форму. При этом обе
схемы сохраняют $M$ с точностью округления. Поэтому наряду с балансом нужен
контроль ошибки относительно точного решения. Таблица позволяет отдельно
проверить первый и второй порядки на гладком профиле.

## 3. Прямоугольный импульс

Повторим расчёты для $C=0.5,0.8,1$ на всех трёх сетках. Отрицательные
значения и значения выше единицы сохраняются в расчётах и на графиках.
Сначала сравним профили при $N=200$, затем измерим ошибку и экстремумы
для всех сеток.

In [ ]:
pulse = {}
for C in CS:
    for N in NS:
        pair = simulate(N, C, "pulse")
        for method in METHODS:
            pulse[method, N, C] = pair[method]

fig, axes = plt.subplots(2, 3, figsize=(13, 6.5), sharex=True, sharey=True,
                         layout="constrained")
for row, target in enumerate(TIMES):
    for col, C in enumerate(CS):
        ax = axes[row, col]
        ax.plot(xf, exact_solution(xf, target, "pulse"), "k--", lw=1.4,
                drawstyle="steps-post", label="Точное решение")
        for method in METHODS:
            run = pulse[method, 200, C]
            k = at_time(run, target)
            ax.plot(run["x"], run["u"][k], color=COLORS[method], label=LABELS[method])
        ax.set(title=f"C = {C:g}, t = {target:g}", xlim=(0, 1), ylim=(-0.3, 1.3))
        if row == 1:
            ax.set_xlabel("x")
        if col == 0:
            ax.set_ylabel("u")
axes[0, 0].legend(fontsize=8)
fig.suptitle("Прямоугольный импульс, N = 200")
plt.show()

### Фронты в увеличенном масштабе

Показан один оборот при $N=200$, $C=0.8$. Ширина каждого окна равна $0.24$,
вертикальные масштабы одинаковы. Upwind сглаживает оба фронта, а схема
Лакса–Вендроффа создаёт колебания рядом с разрывами.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4), sharey=True, layout="constrained")
for ax, edge, title in zip(axes, (0.2, 0.4), ("Левый фронт: 0 → 1", "Правый фронт: 1 → 0")):
    ax.plot(xf, exact_solution(xf, 1.0, "pulse"), "k--", lw=1.5,
            drawstyle="steps-post", label="Точное решение")
    for method in METHODS:
        run = pulse[method, 200, 0.8]
        k = at_time(run, 1.0)
        ax.plot(run["x"], run["u"][k], "o-", ms=2.8,
                color=COLORS[method], label=LABELS[method])
    ax.set(title=title, xlabel="x", xlim=(edge - 0.12, edge + 0.12), ylim=(-0.25, 1.25))
    ax.axhline(0, color="0.5", lw=0.7)
    ax.axhline(1, color="0.5", lw=0.7)
axes[0].set_ylabel("u")
axes[0].legend(fontsize=9)
fig.suptitle("Оба фронта после одного оборота: N = 200, C = 0.8")
plt.show()

### Ошибка, минимум и максимум при сгущении сетки

Для разрывного профиля гладкость, необходимая для второго порядка
Лакса–Вендроффа, отсутствует. Поэтому не переносим на него оценку порядка,
полученную для синусоиды. В таблицах оставляем также $C=1$; на графиках
ошибки от $\Delta x$ показываем $C=0.5$ и $0.8$, поскольку при $C=1$
ошибка находится на уровне округления.

In [ ]:
for target in TIMES:
    rows = []
    for C in CS:
        for method in METHODS:
            for N in NS:
                run = pulse[method, N, C]
                k = at_time(run, target)
                rows.append([f"{C:g}", LABELS[method], N, f"{run['E1'][k]:.5e}",
                             f"{run['min'][k]:.5f}", f"{run['max'][k]:.5f}"])
    title = "После одного оборота" if target == 1 else "После пяти оборотов"
    display(Markdown(f"**{title}**"))
    table(["C", "Схема", "N", "E₁", "min u", "max u"], rows)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4), layout="constrained")
for ax, target in zip(axes, TIMES):
    for C, style in ((0.5, "o-"), (0.8, "s--")):
        for method in METHODS:
            errors = [pulse[method, N, C]["E1"][at_time(pulse[method, N, C], target)]
                      for N in NS]
            ax.loglog(dx, errors, style, color=COLORS[method],
                      label=f"{LABELS[method]}, C = {C:g}")
    ax.set(title=f"Прямоугольный импульс, t = {target:g}", xlabel="Δx", ylabel="E₁")
    ax.legend(fontsize=8)
plt.show()

### Изменение экстремумов и ошибки во времени

Сравним все устойчивые числа Куранта при $N=100$. Минимум и максимум
показаны в общей для всех панелей шкале. Ошибка вычисляется относительно
точного сдвига на том же временном слое. Для прямоугольника узловая
дискретизация точного фронта может давать небольшие колебания $E_1(t)$;
это не означает неустойчивости. В моменты целого числа оборотов фронты
снова лежат в исходных узлах.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(13, 8), sharex=True, sharey="row",
                         layout="constrained")
for col, C in enumerate(CS):
    for method in METHODS:
        run = pulse[method, 100, C]
        for row, key in enumerate(("min", "max", "E1")):
            axes[row, col].plot(run["t"], run[key], color=COLORS[method], label=LABELS[method])
    axes[0, col].set_title(f"C = {C:g}")
    axes[0, col].axhline(0, color="black", ls=":", lw=1)
    axes[1, col].axhline(1, color="black", ls=":", lw=1)
    axes[0, col].set_ylim(-0.3, 0.08)
    axes[1, col].set_ylim(0.1, 1.3)
    axes[2, col].set_ylim(-0.005, 0.35)
    axes[2, col].set_xlabel("t")
    axes[0, col].legend(fontsize=8)
for row, label in enumerate(("min u", "max u", "E₁")):
    axes[row, 0].set_ylabel(label)
fig.suptitle("Устойчивые расчёты, прямоугольный импульс, N = 100")
plt.show()

rows = []
for C in CS:
    for method in METHODS:
        drift = max(np.max(np.abs(pulse[method, N, C]["M"] - pulse[method, N, C]["M"][0]))
                    for N in NS)
        rows.append([f"{C:g}", LABELS[method], f"{drift:.3e}"])
table(["C", "Схема", "max |M − M₀| по трём сеткам и 0 ≤ t ≤ 5"], rows)

### Почему возникают выбросы и чем выделяется $C=1$?

У upwind при $0<C<1$ ведущая диффузионная поправка имеет коэффициент
$D_{\rm num}=c\Delta x(1-C)/2$. Поэтому фронты размазываются; увеличение
$N$ или приближение $C$ к единице ослабляет это размытие.

Схему Лакса–Вендроффа можно записать как
$$
u_j^{n+1}=\frac{C(C+1)}2u_{j-1}^n+(1-C^2)u_j^n
+\frac{C(C-1)}2u_{j+1}^n.
$$
При $0<C<1$ последний коэффициент отрицателен. Схема не является
выпуклым усреднением и может давать $u<0$ и $u>1$ рядом с разрывом.
Второй порядок на гладких решениях не гарантирует монотонность на скачке.
Сгущение сетки уменьшает интегральную ошибку, но само по себе не устраняет
осцилляции: их высота на исследованных сетках не обязана монотонно убывать.

При $C=1$ обе формулы превращаются в $u_j^{n+1}=u_{j-1}^n$.
За шаг решение сдвигается ровно на один узел, а за $N$ шагов возвращается
на исходное место. Это точный перенос **дискретного** профиля; положение
неразрешённого сеткой фронта между узлами от этого не уточняется.
В наших опытах $N$, $t=1$ и $t=5$ согласованы с целым числом шагов,
поэтому отличия от нуля в таблице для $C=1$ связаны с округлением.

## 4. Нарушение CFL: $C=1.1$

Для отдельного опыта берём прямоугольный импульс, $N=100$ и $0\le t\le1$.
Уже этого времени достаточно, чтобы увидеть рост неустойчивости.

Для гармоники $u_j^n=G^n e^{i j\theta}$ коэффициенты усиления равны
$$
G_{\rm up}=1-C(1-e^{-i\theta}),\qquad
G_{\rm LW}=1-iC\sin\theta+C^2(\cos\theta-1).
$$
При $\theta=\pi$ получаем $|G_{\rm up}|=|1-2C|$ и
$|G_{\rm LW}|=|1-2C^2|$. Для $C=1.1$ это $1.2$ и $1.42$:
высокочастотные возмущения усиливаются. У выбранного прямоугольника
конкретно шахматная гармоника может отсутствовать, но соседние
высокочастотные моды также неустойчивы и присутствуют в начальных данных.

In [ ]:
unstable = simulate(100, 1.1, "pulse", targets=(0.25, 0.5, 1.0))

fig, axes = plt.subplots(2, 3, figsize=(13, 6.5), sharex=True, layout="constrained")
for row, method in enumerate(METHODS):
    run = unstable[method]
    for col, target in enumerate((0.25, 0.5, 1.0)):
        ax = axes[row, col]
        k = at_time(run, target)
        ax.plot(xf, exact_solution(xf, target, "pulse"), "k--", lw=1,
                drawstyle="steps-post", label="Точное решение")
        ax.plot(run["x"], run["u"][k], color=COLORS[method], label=LABELS[method])
        ax.set(title=f"{LABELS[method]}, t = {target:g}", xlim=(0, 1))
        ax.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
        if col == 0:
            ax.set_ylabel("u")
        if row == 1:
            ax.set_xlabel("x")
axes[0, 0].legend(fontsize=8)
fig.suptitle("Неустойчивый расчёт: C = 1.1, N = 100; вертикальные масштабы различаются")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), layout="constrained")
for method in METHODS:
    run = unstable[method]
    axes[0].plot(run["t"], run["min"], color=COLORS[method], label=LABELS[method])
    axes[1].plot(run["t"], run["max"], color=COLORS[method], label=LABELS[method])
    mask = run["E1"] > 0
    axes[2].semilogy(run["t"][mask], run["E1"][mask], color=COLORS[method], label=LABELS[method])
for ax, label in zip(axes, ("min u", "max u", "E₁")):
    ax.set(xlabel="t", ylabel=label)
    ax.legend(fontsize=8)
for ax in axes[:2]:
    ax.set_yscale("symlog", linthresh=1)
fig.suptitle("Рост экстремумов и ошибки при C = 1.1, N = 100")
plt.show()

table(["Схема", "E₁(1)", "min u(1)", "max u(1)"], [
    [LABELS[m], f"{unstable[m]['E1'][-1]:.4e}",
     f"{unstable[m]['min'][-1]:.4e}", f"{unstable[m]['max'][-1]:.4e}"]
    for m in METHODS
])

Устойчивые расчёты с $C<1$ и опыт с $C=1.1$ отличаются не только высотой
выбросов. В устойчивой схеме Лакса–Вендроффа осцилляции остаются
ограниченными, а здесь ошибка и экстремумы быстро растут на порядки.
Формальное сохранение суммы значений не устраняет эту неустойчивость.

## 5. Анимация переноса

Показаны точное решение и две схемы для $N=100$, $C=0.8$ от $t=0$ до $t=5$.
Обе панели используют один временной слой: сверху — профили, снизу —
накопленная история $E_1$ и текущий момент времени. Через периодическую
границу профиль проходит без специальных граничных условий: её учитывает
циклическая индексация.

In [ ]:
run_up = pulse["upwind", 100, 0.8]
run_lw = pulse["LW", 100, 0.8]
frame_ids = np.unique(np.r_[np.linspace(0, len(run_up["t"]) - 1, 121, dtype=int),
                                at_time(run_up, 1.0)])

fig_anim, (ax_u, ax_e) = plt.subplots(2, 1, figsize=(10, 6),
                                    gridspec_kw={"height_ratios": [2, 1]},
                                    layout="constrained")
exact_line, = ax_u.plot([], [], "k--", lw=1.4, drawstyle="steps-post", label="Точное решение")
up_line, = ax_u.plot([], [], color=BLUE, label="Upwind")
lw_line, = ax_u.plot([], [], color=ORANGE, label="Лакс–Вендрофф")
ax_u.set(xlim=(0, 1), ylim=(-0.3, 1.3), xlabel="x", ylabel="u")
ax_u.legend(loc="lower center", bbox_to_anchor=(0.5, 1.10), ncol=3, fontsize=9)
time_text = ax_u.set_title("")
for run, method in ((run_up, "upwind"), (run_lw, "LW")):
    ax_e.plot(run["t"], run["E1"], color=COLORS[method], alpha=0.3, lw=1)
history_up, = ax_e.plot([], [], color=BLUE)
history_lw, = ax_e.plot([], [], color=ORANGE)
time_line = ax_e.axvline(0, color="0.3", ls=":")
error_limit = 1.10 * max(run_up["E1"].max(), run_lw["E1"].max())
ax_e.set(xlim=(0, 5), ylim=(0, error_limit), xlabel="t", ylabel="E₁")


def animate(frame):
    k = frame_ids[frame]
    t = run_up["t"][k]
    exact_line.set_data(xf, exact_solution(xf, t, "pulse"))
    up_line.set_data(run_up["x"], run_up["u"][k])
    lw_line.set_data(run_lw["x"], run_lw["u"][k])
    time_text.set_text(f"t = {t:.3f}; N = 100; C = 0.8")
    history_up.set_data(run_up["t"][:k+1], run_up["E1"][:k+1])
    history_lw.set_data(run_lw["t"][:k+1], run_lw["E1"][:k+1])
    time_line.set_xdata([t, t])
    return exact_line, up_line, lw_line, history_up, history_lw, time_line, time_text


anim = FuncAnimation(fig_anim, animate, frames=len(frame_ids), interval=80, blit=True)
animation_html = HTML(anim.to_jshtml(default_mode="once"))
plt.close(fig_anim)
display(animation_html)

## 6. Проверка расчётов и выводы

Проверим попадание в заданные моменты, ограничение на шаг, сохранение
количества примеси, отсутствие новых экстремумов у устойчивой upwind-схемы
и точный сеточный сдвиг при $C=1$. Сходимость гладкого профиля дополнительно
проверим по независимой формуле усиления одной гармоники.

In [ ]:
# Сравнение с независимым расчётом эволюции единственной гармоники.
fourier_difference = 0.0
for (method, N), run in smooth.items():
    theta = 2 * np.pi / N
    nu = N * np.diff(run["t"])
    if method == "upwind":
        factors = 1 - nu * (1 - np.exp(-1j * theta))
    else:
        factors = 1 - 1j * nu * np.sin(theta) + nu**2 * (np.cos(theta) - 1)
    amplitude = np.prod(factors)
    predicted = 1 + 0.5 * np.imag(np.exp(2j * np.pi * run["x"]) * amplitude)
    fourier_difference = max(fourier_difference, np.max(np.abs(predicted - run["u"][-1])))
assert fourier_difference < 1e-11

all_stable = list(smooth.values()) + list(pulse.values())
mass_drift = max(np.max(np.abs(r["M"] - r["M"][0])) for r in all_stable)
assert mass_drift < 1e-12

for (method, N), run in smooth.items():
    assert np.max(N * np.diff(run["t"])) <= 0.8 + 1e-12
    for target in TIMES:
        assert run["t"][at_time(run, target)] == target
for (method, N, C), run in pulse.items():
    assert np.all(np.isfinite(run["u"]))
    assert np.max(N * np.diff(run["t"])) <= C + 1e-12
    if method == "upwind":
        assert run["min"].min() > -1e-12 and run["max"].max() < 1 + 1e-12
    if C == 1:
        assert run["E1"].max() < 1e-11
    for target in TIMES:
        assert run["t"][at_time(run, target)] == target

for method in METHODS:
    for target in TIMES:
        errors = [smooth[method, N]["E1"][at_time(smooth[method, N], target)] for N in NS]
        assert errors[2] < errors[1] < errors[0]
    assert unstable[method]["E1"][-1] > 100

table(["Проверка", "Результат"], [
    ["Максимальный дрейф M во всех устойчивых опытах", f"{mass_drift:.3e}"],
    ["Отличие гладкого решения от формулы Фурье при t = 5", f"{fourier_difference:.3e}"],
    ["Времена сравнения, CFL, монотонность upwind, сдвиг при C = 1", "Проверки пройдены"],
])

1. **На гладком профиле** уменьшение шага сетки снижает ошибку обеих схем.
   Порядок, измеренный по $E_1$, составляет $0.87$–$0.99$ для upwind и
   $2.00$–$2.01$ для Лакса–Вендроффа. Через пять оборотов на сетке $N=200$
   ошибки равны соответственно $2.99\cdot10^{-2}$ и $5.92\cdot10^{-4}$.
   Через пять оборотов ошибка больше, чем через один.
   Количество примеси при этом сохраняется; правильный баланс не означает
   правильного распределения.
2. **На прямоугольном импульсе** upwind сохраняет диапазон $[0,1]$, но
   размывает фронты и уменьшает высоту импульса. Лакс–Вендрофф лучше
   разрешает фронты, однако даёт отрицательные значения и выбросы выше
   единицы. При сгущении сетки интегральная ошибка уменьшается, а высота
   осцилляций не служит надёжным показателем сходимости.
3. **Число Куранта влияет на результат.** При $C=0.8$ upwind размывает
   импульс слабее, чем при $C=0.5$. При $C=1$ обе схемы точно сдвигают
   сеточный профиль на один узел за шаг; этот специальный случай нельзя
   использовать для оценки обычного порядка точности.
4. **При $C=1.1$** обе схемы неустойчивы: экстремумы и ошибка быстро растут.
   Это отличается от ограниченных осцилляций устойчивой схемы
   Лакса–Вендроффа при $C<1$.